In [1]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P0DTD1.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P0DTD1.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [2]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P23946.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P23946.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [3]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "Q9Y5N1.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: Q9Y5N1.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [4]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "Q13822.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: Q13822.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [5]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "Q9UNE7.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: Q9UNE7.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [6]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P00533.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P00533.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [7]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "Q16539.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: Q16539.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [8]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/home/nakib/prodrug/prodrug"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "Q07869.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: Q07869.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7fcb741d73a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7fcb741d71f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ❌ FAIL
  Pose 4: ✅ PASS
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ✅ PASS
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 4

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [1]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P00918.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P00918.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [2]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P00918.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P00918.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [3]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Your working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P04083.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P04083.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [5]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P27487.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P27487.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [6]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P27487.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P27487.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [7]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P35354.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P35354.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 8

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 8, Passed: 8
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [8]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P35354.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P35354.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [9]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P23219.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P23219.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [10]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P09917.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P09917.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [11]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P11511.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P11511.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [13]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P05067.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P05067.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [14]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P29274.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P29274.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [15]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P29274.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P29274.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [16]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P08172.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P08172.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [17]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P28223.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P28223.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ❌

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [18]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P30939.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P30939.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [19]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P06493.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P06493.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [20]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P06493.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P06493.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [21]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P53350.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P53350.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [22]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P53350.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P53350.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL VINA POSES PASSED VALIDATION!

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

✅✅✅ ALL GNINA POSES PASSED VALIDATION!

FINAL COMPARISON: VINA vs GNINA
  VINA:  ✅ ALL PASSED
  GNINA: ✅ ALL PASSED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 9
  GNINA - Total poses: 9, Passed: 9

  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!


In [23]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P03372.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P03372.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ❌
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ❌ FAIL
  Pose 9: ✅ PASS

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ❌
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ❌ FAIL
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 8
  GNINA - Total poses: 9, Passed: 8

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [24]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P03372.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P03372.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ❌
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ✅ PASS
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ❌ FAIL
  Pose 9: ✅ PASS

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f14243633a0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f14243631f0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ❌
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ✅ PASS
  Pose 2: ✅ PASS
  Pose 3: ✅ PASS
  Pose 4: ❌ FAIL
  Pose 5: ✅ PASS
  Pose 6: ✅ PASS
  Pose 7: ✅ PASS
  Pose 8: ✅ PASS
  Pose 9: ✅ PASS

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 8
  GNINA - Total poses: 9, Passed: 8

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


<h1>corrected 4</h1>

In [2]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P27487.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P27487.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters


In [3]:
# ============================================
# POSEBUSTERS VALIDATION 
# ============================================

import subprocess
from posebusters import PoseBusters
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================

# Working directory
WORK_DIR = "/mnt/data/nakib/prodrug/prodrug/"

# Example 2 specific files
protein_file = os.path.join(WORK_DIR, "P06493.pdb")           # From Step 1
vina_pdbqt = os.path.join(WORK_DIR, "vina_results.pdbqt")     # From Step 3 (Vina)
gnina_pdbqt = os.path.join(WORK_DIR, "gnina_results.pdbqt")   # From Step 3 (Gnina)
vina_sdf = "vina_ligand.sdf"                         # Temporary output for Vina
gnina_sdf = "gnina_ligand.sdf"                       # Temporary output for Gnina

print("="*70)
print("POSEBUSTERS VALIDATION")
print("="*70)
print(f"Protein: ")

print("="*70)

# ============================================
# FUNCTION TO VALIDATE A DOCKING RESULT
# ============================================

def validate_docking(ligand_pdbqt, ligand_sdf, engine_name):
    """Run PoseBusters validation for a given docking result"""
    
    print(f"\n{'='*70}")
    print(f"VALIDATING {engine_name} DOCKING RESULTS")
    print("="*70)
    
    # Check if file exists
    if not os.path.exists(ligand_pdbqt):
        print(f"  ❌ Ligand file NOT found: {os.path.basename(ligand_pdbqt)}")
        return None
    
    print(f"  ✅ Ligand file found: {os.path.basename(ligand_pdbqt)}")
    
    # Convert PDBQT to SDF
    print(f"\n🔄 Converting {engine_name} PDBQT → SDF...")
    try:
        subprocess.run(["obabel", ligand_pdbqt, "-O", ligand_sdf], check=True, 
                       capture_output=True, text=True)
        print("  ✅ Conversion successful")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        return None
    
    # Run PoseBusters validation
    print(f"\n🔬 Running PoseBusters validation (22 checks) for {engine_name}...")
    bust = PoseBusters(config="dock")
    
    try:
        results = bust.bust(
            mol_pred=ligand_sdf,
            mol_cond=protein_file
        )
        
        # Convert results to dataframe
        df = pd.DataFrame(results)
        
        print(f"\n📊 {engine_name} VALIDATION RESULTS:")
        print("="*70)
        
        # Display key checks
        key_checks = ['sanitization', 'bond_lengths', 'bond_angles', 
                      'aromatic_ring_flatness', 'internal_steric_clash',
                      'protein-ligand_maximum_distance', 'volume_overlap_with_protein']
        
        for check in key_checks:
            if check in df.columns:
                passed = df[check].all()
                status = '✅' if passed else '❌'
                print(f"  {check:30}: {status}")
        
        # Summary
        print("\n" + "="*70)
        print(f"{engine_name} DOCKING VALIDATION SUMMARY")
        print("="*70)
        
        total_poses = len(df)
        print(f"Total docking poses analyzed: {total_poses}")
        
        # Check passes (ignore no_radicals warning)
        critical_cols = [c for c in df.columns if c != "no_radicals"]
        passes = df[critical_cols].all(axis=1)
        
        print("\nPose Status:")
        for i, status in enumerate(passes):
            if status:
                print(f"  Pose {i+1}: ✅ PASS")
            else:
                print(f"  Pose {i+1}: ❌ FAIL")
        
        print("\n" + "="*70)
        if passes.all():
            print(f"✅✅✅ ALL {engine_name} POSES PASSED VALIDATION!")
        else:
            print(f"⚠️  Some {engine_name} poses failed validation checks.")
            print("   Review the results above for details.")
        print("="*70)
        
        # Clean up temporary SDF file
        if os.path.exists(ligand_sdf):
            os.remove(ligand_sdf)
        
        return {
            'engine': engine_name,
            'total_poses': total_poses,
            'passes': passes,
            'all_passed': passes.all(),
            'dataframe': df
        }
        
    except Exception as e:
        print(f"❌ Error during validation for {engine_name}: {e}")
        return None

# ============================================
# MAIN VALIDATION
# ============================================

# Check if protein file exists
print("\n🔍 Checking protein file...")
if os.path.exists(protein_file):
    print(f"  ✅ Protein file found: {os.path.basename(protein_file)}")
else:
    print(f"  ❌ Protein file NOT found: {protein_file}")
    exit()

# Validate Vina results
vina_results = validate_docking(vina_pdbqt, vina_sdf, "VINA")

# Validate Gnina results
gnina_results = validate_docking(gnina_pdbqt, gnina_sdf, "GNINA")

# ============================================
# FINAL COMPARISON
# ============================================

print("\n" + "="*70)
print("FINAL COMPARISON: VINA vs GNINA")
print("="*70)

if vina_results:
    print(f"  VINA:  {'✅ ALL PASSED' if vina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  VINA:  ❌ VALIDATION FAILED")

if gnina_results:
    print(f"  GNINA: {'✅ ALL PASSED' if gnina_results['all_passed'] else '❌ SOME FAILED'}")
else:
    print(f"  GNINA: ❌ VALIDATION FAILED")

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE")
print("="*70)

# Display detailed comparison if both succeeded
if vina_results and gnina_results:
    print("\n📊 Detailed Comparison:")
    print(f"  VINA  - Total poses: {vina_results['total_poses']}, Passed: {sum(vina_results['passes'])}")
    print(f"  GNINA - Total poses: {gnina_results['total_poses']}, Passed: {sum(gnina_results['passes'])}")
    
    if vina_results['all_passed'] and gnina_results['all_passed']:
        print("\n  ✅✅✅ BOTH DOCKING ENGINES PRODUCED VALID POSES!")
    elif vina_results['all_passed']:
        print("\n  ✅ VINA produced valid poses, GNINA had issues")
    elif gnina_results['all_passed']:
        print("\n  ✅ GNINA produced valid poses, VINA had issues")
    else:
        print("\n  ⚠️ BOTH engines produced some invalid poses - review docking parameters")

/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})


POSEBUSTERS VALIDATION
Protein: 

🔍 Checking protein file...
  ✅ Protein file found: P06493.pdb

VALIDATING VINA DOCKING RESULTS
  ✅ Ligand file found: vina_results.pdbqt

🔄 Converting VINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for VINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 VINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

VINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some VINA poses failed validation checks.
   Review the results above for details.

VALIDATING GNINA DOCKING RESULTS
  ✅ Ligand file found: gnina_results.pdbqt

🔄 Converting GNINA PDBQT → SDF...
  ✅ Conversion successful

🔬 Running PoseBusters validation (22 checks) for GNINA...


/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amin at 0x7f11a808ee50> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/posebusters/modules/distance_geometry.py:100: FutureWarning: The provided callable <function amax at 0x7f11a808eca0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
/home/nakib/miniconda3/envs/posebusters_env/lib/python3.9/site-packages/pose


📊 GNINA VALIDATION RESULTS:
  sanitization                  : ✅
  bond_lengths                  : ✅
  bond_angles                   : ✅
  aromatic_ring_flatness        : ✅
  internal_steric_clash         : ✅
  protein-ligand_maximum_distance: ✅
  volume_overlap_with_protein   : ✅

GNINA DOCKING VALIDATION SUMMARY
Total docking poses analyzed: 9

Pose Status:
  Pose 1: ❌ FAIL
  Pose 2: ❌ FAIL
  Pose 3: ❌ FAIL
  Pose 4: ❌ FAIL
  Pose 5: ❌ FAIL
  Pose 6: ❌ FAIL
  Pose 7: ❌ FAIL
  Pose 8: ❌ FAIL
  Pose 9: ❌ FAIL

⚠️  Some GNINA poses failed validation checks.
   Review the results above for details.

FINAL COMPARISON: VINA vs GNINA
  VINA:  ❌ SOME FAILED
  GNINA: ❌ SOME FAILED

✅ VALIDATION COMPLETE

📊 Detailed Comparison:
  VINA  - Total poses: 9, Passed: 0
  GNINA - Total poses: 9, Passed: 0

  ⚠️ BOTH engines produced some invalid poses - review docking parameters
